# 9장. 점수를 비율로, 숫자를 바꾸는 부품

점수를 비율로 바꾸는 softmax, 뒤 글자를 가리는 마스크, 그리고 LLM을 이루는 부품 nn.Linear와 nn.Embedding, 부품을 만드는 틀인 클래스를 직접 써 봅니다.

- GPU가 없어도 됩니다.
- 셀은 위에서부터 차례로 실행합니다(Shift + Enter).

## 실습 9-1 점수를 비율로: softmax

"가"의 점수 [4, 2, 0]을 비율로 바꿉니다.

In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm-v2
import torch
from torch import nn
from torch.nn import functional as F
from modu_llm import Vocab

s = torch.tensor([4.0, 2.0, 0.0])      # "가"의 점수
ex = torch.exp(s)
print("지수 함수로 키우기:", ex)
print("합으로 나누기:", ex / ex.sum())
print("softmax:", F.softmax(s, dim=-1))

## 실습 9-2 쏠리는 비율, 나눠서 고르게

**먼저 예상해 보세요.** 점수를 10배로 키우면 비율은 고르게 퍼질까요, 한쪽으로 몰릴까요?

In [ ]:
print("점수 10배:", F.softmax(s * 10, dim=-1).round(decimals=2))
print("점수를 2로 나누기:", F.softmax(s / 2, dim=-1))

## 실습 9-3 뒤 글자 가리기: 마스크

세 글자의 점수 표에서 뒤 글자 자리를 가린 뒤 비율 표를 만듭니다.

In [ ]:
S = torch.tensor([[0.0, 0.0, 2.0],      # "토"의 점수
                  [2.0, 0.0, -2.0],     # "끼"의 점수
                  [4.0, 2.0, 0.0]])     # "가"의 점수
mask = torch.tril(torch.ones(3, 3))
print(mask)

masked = (S / 2).masked_fill(mask == 0, float("-inf"))
print(masked)
W = F.softmax(masked, dim=-1)
print(W.round(decimals=2))
print("줄마다 합:", W.sum(dim=-1))

## 실습 9-4 곱하고 더하는 표

글자 두 개의 숫자 3개를 숫자 2개로 바꿉니다. 손으로 계산한 값과 비교해 보세요.

In [ ]:
x = torch.tensor([[1.0, 2.0, 0.0],      # 글자 A의 숫자 3개
                  [0.0, 1.0, 3.0]])     # 글자 B의 숫자 3개
R = torch.tensor([[0.5, 1.0, 0.0],      # 출력1의 레시피
                  [-1.0, 0.0, 1.0]])    # 출력2의 레시피
print(x @ R.T)

## 실습 9-5 부품 nn.Linear

같은 일을 하는 PyTorch 부품입니다. 레시피(가중치)는 무작위로 정해집니다.

In [ ]:
torch.manual_seed(0)
linear = nn.Linear(3, 2, bias=False)
print("가중치 모양:", linear.weight.shape)
print(linear(x))
print("x @ 가중치.T와 같은가요?", torch.allclose(linear(x), x @ linear.weight.T))

## 실습 9-6 부품 nn.Embedding

글자 번호로 숫자 카드를 꺼내는 표입니다.

**먼저 예상해 보세요.** "토끼가 숲에 가요"에는 '가'가 두 번 나옵니다. 두 '가'의 숫자 카드는 같을까요?

In [ ]:
vocab = Vocab.build()
torch.manual_seed(0)
emb = nn.Embedding(len(vocab), 8)
print("표 모양:", emb.weight.shape)

ids = torch.tensor(vocab.encode("토끼가 숲에 가요"))
cards = emb(ids)
print("숫자 카드 모양:", cards.shape)
print("표의 줄을 꺼낸 것과 같은가요?", torch.equal(cards[0], emb.weight[ids[0]]))
print("두 '가'의 카드가 같은가요?", torch.equal(cards[2], cards[7]))

## 실습 9-7 부품을 만드는 틀: 클래스

붕어빵 틀로 클래스의 모양을 익힙니다.

In [ ]:
class FishBread:
    def __init__(self, filling):
        self.filling = filling

    def describe(self):
        return self.filling + " 붕어빵"

bread1 = FishBread("팥")
bread2 = FishBread("슈크림")
print(bread1.describe())
print(bread2.describe())
print(bread1.filling)

## 실습 9-8 비율 표를 만드는 부품

이 장에서 배운 것을 모아, 숫자 카드를 넣으면 비율 표를 계산하는 부품을 만듭니다.

In [ ]:
class RatioTable(nn.Module):
    def __init__(self, n_embd, head_size):
        super().__init__()
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.head_size = head_size

    def forward(self, x):
        q = self.query(x)                               # 질문 쪽지
        k = self.key(x)                                 # 명찰
        scores = q @ k.T / self.head_size ** 0.5        # 점수와 나누기
        T = x.shape[0]
        mask = torch.tril(torch.ones(T, T))
        scores = scores.masked_fill(mask == 0, float("-inf"))   # 가리기
        return F.softmax(scores, dim=-1)                # 비율

torch.manual_seed(0)
ratio = RatioTable(8, 4)
W9 = ratio(cards)
print("비율 표 모양:", W9.shape)
print(W9.round(decimals=2))
print("줄마다 합:", W9.sum(dim=-1))